# Tutorial 6 — Attention on Structured Geometric Data

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 6: Transformers and Attention**

---

A convolution (Tutorial 5) fixes its neighbourhood in advance: pixel $(i,j)$ talks to
$(i\pm1, j\pm1)$, always, because the grid says so. Much geometric data has no grid.
A point cloud, a set of simplices, the vertices of a triangulation — these are
**sets**, and the only structure available is what the data itself supplies.

Attention is the operator for that situation. It computes, from the data, *which
elements should talk to which*, and then averages accordingly. Its native symmetry is
not translation but **permutation**: relabel the elements and the output relabels
with them.

| § | Question |
|---|---|
| 1 | Attention as a permutation-equivariant operator on a set |
| 2 | A geometric task where order is meaningless and size varies |
| 3 | A transformer that never pads vs a padded MLP that sees an ordering |
| 4 | Reading the attention weights: what does the model look at? |
| 5 | Summary |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 20260910
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. Attention, and the symmetry it respects

Given $n$ elements with feature vectors $x_1,\dots,x_n \in \mathbb{R}^d$, stacked as
$X \in \mathbb{R}^{n\times d}$, **scaled dot-product attention** forms three linear
images of the data and mixes them:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V,
\qquad
\mathrm{Att}(X) \;=\; \underbrace{\mathrm{softmax}\!\Big(\tfrac{QK^\top}{\sqrt{d_k}}\Big)}_{A \;\in\; \mathbb{R}^{n\times n}} V .$$

The matrix $A$ is row-stochastic: row $i$ is a probability distribution over the
other elements, saying how much element $i$ attends to each. Two things follow.

- **Attention is a data-dependent averaging operator.** Convolution also averages
  neighbours, but with weights fixed by the architecture; here $A$ is computed from
  $X$ and changes with every input. This is the sense in which the neighbourhood is
  *learned* rather than declared.
- **Attention is permutation-equivariant.** For a permutation matrix $P$,
  $\mathrm{Att}(PX) = P\,\mathrm{Att}(X)$: the entries of $A$ are functions of pairs
  $(x_i, x_j)$ only, so permuting the input permutes the rows and columns of $A$ in
  step. There is nothing in the formula that knows about order.

That last point is why attention suits sets. It is also why transformers need
*positional encoding* when order does matter — the mechanism has no access to it
otherwise, and it must be supplied as part of the features.

In [ ]:
def attention(X, Wq, Wk, Wv, return_weights=False):
    """Scaled dot-product attention, written out."""
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.transpose(-2, -1) / np.sqrt(K.shape[-1])
    A = torch.softmax(scores, dim=-1)
    out = A @ V
    return (out, A) if return_weights else out


n, d, dk = 6, 4, 8
X = torch.tensor(rng.normal(size=(n, d)))
Wq, Wk, Wv = (torch.tensor(rng.normal(size=(d, dk)) * 0.5) for _ in range(3))

perm = torch.tensor(rng.permutation(n))
P = torch.eye(n, dtype=torch.float64)[perm]

out, A = attention(X, Wq, Wk, Wv, return_weights=True)
out_perm = attention(P @ X, Wq, Wk, Wv)

print(f"rows of A sum to 1:            {A.sum(dim=1).numpy().round(12)}")
print(f"permutation equivariance:      ||Att(PX) - P Att(X)||_inf = "
      f"{(out_perm - P @ out).abs().max():.2e}")
print(f"a global mean gives invariance: ||mean Att(PX) - mean Att(X)||_inf = "
      f"{(out_perm.mean(0) - out.mean(0)).abs().max():.2e}")

---
## 2. A task where order is meaningless

We need a label that depends on a point cloud as a **set**, not as a list. Take $n$
points sampled in the plane and ask for the **diameter**,

$$\mathrm{diam}(S) \;=\; \max_{i,j} \lVert x_i - x_j \rVert ,$$

which is exact, cheap to compute, permutation-invariant, and — importantly —
depends on only two of the $n$ points. A model must *find* the relevant pair, which
is precisely the kind of relational question attention is built for.

The clouds are sampled from random ellipses, so the extreme pair is usually well
separated from the rest. **Each cloud has its own number of points**, drawn between
10 and 22. A set has a cardinality, not a length, and nothing about the diameter
requires every input to be the same size — attention obliges, since the $n\times n$
matrix $A$ simply takes whatever $n$ the input supplies.

So we store the data as it is: a **list of $(n_i, 2)$ arrays**, no padding, no mask.
Padding is not something attention needs; it is something *rectangular arrays* need,
and it enters a codebase only when you insist on stacking unequal clouds into one
tensor. §3 shows the two ways out — and which model is forced to take which.

One detail matters enormously and is easy to skip past: **the points arrive sorted
by angle**. That is a perfectly natural way for such data to reach you — traversal
order around a boundary, a scan order, the order a solver emitted. It means the
input carries a real, learnable ordering, which is exactly the situation in which
the choice of architecture starts to bite.

In [ ]:
N_MIN, N_MAX = 10, 22          # points per cloud: drawn per cloud, not fixed


def make_clouds(n_sets, rng, n_min=N_MIN, n_max=N_MAX):
    """Point clouds on random ellipses, labelled by their diameter.

    Returns a *list* of (n_i, 2) arrays. Each cloud keeps its own length; nothing is
    padded, and there is no mask to carry around."""
    a = rng.uniform(0.4, 1.0, size=n_sets)
    b = rng.uniform(0.4, 1.0, size=n_sets)
    th = rng.uniform(0, np.pi, size=n_sets)
    clouds, diam = [], np.empty(n_sets, np.float32)
    for i, n in enumerate(rng.integers(n_min, n_max + 1, size=n_sets)):
        ang = np.sort(rng.uniform(0, 2 * np.pi, size=n))
        x, y = a[i] * np.cos(ang), b[i] * np.sin(ang)
        C = np.stack([x * np.cos(th[i]) - y * np.sin(th[i]),
                      x * np.sin(th[i]) + y * np.cos(th[i])], axis=-1)
        C += 0.03 * rng.normal(size=C.shape)
        clouds.append(C.astype(np.float32))
        diam[i] = np.linalg.norm(C[:, None] - C[None, :], axis=-1).max()
    return clouds, diam


def diameter_pair(C):
    d = np.linalg.norm(C[:, None] - C[None, :], axis=-1)
    return np.unravel_index(np.argmax(d), d.shape)


tr, ytr = make_clouds(6000, rng)
te, yte = make_clouds(1500, rng)
lens = np.array([len(c) for c in tr])
print(f"{len(tr)} training clouds, each its own array: {lens.min()}-{lens.max()} points "
      f"(mean {lens.mean():.1f}), nothing padded")
print(f"diameter in [{ytr.min():.2f}, {ytr.max():.2f}]")
print(f"predict-the-mean baseline MSE: {np.mean((yte - ytr.mean())**2):.4f}")

fig, axes = plt.subplots(1, 5, figsize=(11.0, 2.4))
for j, ax in enumerate(axes):
    C = te[j]
    i, jj = diameter_pair(C)
    ax.scatter(*C.T, s=18, color=GEO_DARK)
    ax.plot(C[[i, jj], 0], C[[i, jj], 1], color=GEO_RUST, lw=1.6)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"n = {len(C)},  diam = {yte[j]:.2f}", fontsize=8)
fig.suptitle("point clouds of varying size on random ellipses; the diameter pair in rust", y=1.06)
plt.tight_layout(); plt.show()

---
## 3. Attention versus a model that sees an ordering

Two architectures, same data:

- **Set transformer.** The block of Lecture 6, written out: embed each point, then
  $z = x + \mathrm{Att}(\mathrm{LN}(x))$ and $x' = z + \mathrm{MLP}(\mathrm{LN}(z))$,
  with the MLP applied **tokenwise** — the same small network on each point separately,
  so attention is the only place where points meet. Finish with a **mean over tokens**,
  which contracts the point dimension and leaves one scalar per cloud. Watch where $n$
  goes: it indexes a tensor dimension and the mean removes it, so nothing in the module
  ever names a length. The same weights run on a cloud of any size, and **nothing is
  padded** — at any point.
- **MLP on the flattened cloud.** Concatenate the points into one vector and apply a
  dense network. `nn.Linear` has a fixed input width, so this model *cannot* be
  written without committing to a length — it has to **pad every cloud to
  $n_{\max}$ itself**. The padding lives inside `FlatMLP`, which is where it belongs:
  it is a cost of the architecture, not a property of the data. (It does leak $n$ to
  the model. Worth checking rather than assuming: $n$ and the diameter are nearly
  uncorrelated here, and the best predictor using $n$ alone barely beats predicting
  the mean, so the comparison below is not contaminated by it.)

Batching is the only remaining reason anyone pads, and it is avoidable: group clouds
of **equal $n$** into a batch and every tensor is rectangular already. That is all
`pack_by_length` does. (Production transformers, with lengths spread over thousands,
reach for masks or a varlen kernel instead; at this scale bucketing is exact and
free.)

At test time we evaluate both on the ordinary test set and on a **shuffled** copy in
which every cloud's points have been randomly reordered. The label does not change.

In [ ]:
def sinusoid(n, d):
    """Sinusoidal positional encoding (Lecture 6, slide 5). A *function of the index*,
    not a table with a maximum length, so it is defined for whatever n turns up."""
    pos = torch.arange(n).float().unsqueeze(1)
    ang = pos / (10000 ** (torch.arange(0, d, 2).float() / d))
    return torch.stack([ang.sin(), ang.cos()], dim=-1).flatten(1)


class SetAttention(nn.Module):
    """The transformer block of Lecture 6, written out:

           z  = x + Att(LN(x))
           x' = z + MLP(LN(z))

    The MLP is **tokenwise** — the same small network applied to each point on its own,
    so attention is the only place where points meet. Nothing here names a length: n is
    a tensor dimension, the mean over tokens contracts it, and the same weights run on a
    cloud of any size. There is nothing to pad, at any point.

    (Slide 10 embeds with a single linear layer and reads out with another, but stacks
    three blocks. We keep one block, so the embed and readout each get a hidden layer to
    compensate. The block itself is exactly the form above.)

    positional=True adds the encoding of slide 5 — the only way a tokenwise model can
    see order at all. Unused here; Exercise 3 needs it."""

    def __init__(self, d_model=32, d_ff=64, positional=False):
        super().__init__()
        self.positional = positional
        self.embed = nn.Sequential(nn.Linear(2, d_model), nn.GELU(),
                                   nn.Linear(d_model, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model))          # tokenwise
        self.ln_f = nn.LayerNorm(d_model)
        self.readout = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                     nn.Linear(d_model, 1))

    def forward(self, x, return_A=False):
        h = self.embed(x)                              # (b, n, d_model)
        if self.positional:
            h = h + sinusoid(x.shape[1], h.shape[-1])
        z = self.ln1(h)
        q, k, v = self.Wq(z), self.Wk(z), self.Wv(z)
        A = torch.softmax(q @ k.transpose(1, 2) / np.sqrt(k.shape[-1]), dim=-1)
        h = h + self.Wo(A @ v)                         # z  = x + Att(LN(x))
        h = h + self.mlp(self.ln2(h))                  # x' = z + MLP(LN(z))
        out = self.readout(self.ln_f(h).mean(dim=1))   # mean over tokens -> one
        return (out, A) if return_A else out           #   scalar per cloud


class FlatMLP(nn.Module):
    """The baseline, and what one naively reaches for.

    nn.Linear fixes its input width at construction: a (width, 2*N_MAX) weight can only
    ever multiply a 2*N_MAX vector, and a cloud of n points is 2n numbers. So every
    cloud must be **padded** up to n_max before this model can see it, and a longer one
    cannot be handled at all. That is the cost of per-slot weights."""

    def __init__(self, width=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 * N_MAX, width), nn.ReLU(),
                                 nn.Linear(width, width), nn.ReLU(),
                                 nn.Linear(width, width), nn.ReLU(),
                                 nn.Linear(width, 1))

    def forward(self, x):
        if x.shape[1] > N_MAX:
            raise ValueError(f"needs n <= {N_MAX}, got {x.shape[1]}")
        pad = torch.zeros(len(x), N_MAX - x.shape[1], 2)      # <- the only pad here
        return self.net(torch.cat([x, pad], dim=1).flatten(1))


def pack_by_length(clouds, y=None):
    """Ragged batching: clouds of equal n are stacked together, so every tensor is
    rectangular without a single element of padding."""
    groups = {}
    for i, c in enumerate(clouds):
        groups.setdefault(len(c), []).append(i)
    packs = []
    for idx in groups.values():
        idx = np.array(idx)
        packs.append((torch.tensor(np.stack([clouds[i] for i in idx])),
                      None if y is None else torch.tensor(y[idx]).unsqueeze(1), idx))
    return packs


def train(model, clouds, y, epochs=60, bs=256, lr=3e-3):
    packs = pack_by_length(clouds, y)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    for ep in range(epochs):
        chunks = []
        for X, Y, _ in packs:                     # reshuffle within each length
            q = torch.randperm(len(X)); X, Y = X[q], Y[q]
            chunks += [(X, Y, s) for s in range(0, len(X), bs)]
        for c in rng.permutation(len(chunks)):    # and interleave the lengths
            X, Y, s = chunks[c]
            opt.zero_grad(); lossf(model(X[s:s + bs]), Y[s:s + bs]).backward(); opt.step()
    return model


def predict(model, clouds):
    p = np.empty(len(clouds), np.float32)
    with torch.no_grad():
        for X, _, idx in pack_by_length(clouds):
            p[idx] = model(X).numpy().ravel()
    return p


def mse(model, clouds, y):
    return float(np.mean((predict(model, clouds) - y)**2))


nparams = lambda m: sum(p.numel() for p in m.parameters())

# a shuffled copy of the test set: same sets, different order
te_sh = [c[rng.permutation(len(c))] for c in te]

In [ ]:
torch.manual_seed(0); att = train(SetAttention(), tr, ytr)
torch.manual_seed(0); mlp = train(FlatMLP(), tr, ytr)

base = np.mean((yte - ytr.mean())**2)
print(f"predict-the-mean baseline   MSE {base:.4f}\n")
print(f"  {'model':20s} {'params':>8s} {'pads?':>6s} {'test MSE':>10s} {'shuffled':>11s}")
for name, m, p in [("set transformer", att, "no"), ("MLP on flattened", mlp, "yes")]:
    print(f"  {name:20s} {nparams(m):8,d} {p:>6s} {mse(m, te, yte):10.5f} "
          f"{mse(m, te_sh, yte):11.5f}")

# the same weights on lengths never trained on - and what the padded baseline does
print(f"\n  {'n':>5s} {'transformer':>12s} {'MLP on flattened':>18s} {'true':>8s}")
for n in [len(te[0]), 45, 200]:
    cl, yv = make_clouds(1, rng, n, n)
    xb = torch.tensor(cl[0])[None]
    with torch.no_grad():
        a = f"{att(xb).item():.4f}"
        try:
            b = f"{mlp(xb).item():.4f}"
        except ValueError:
            b = "too large"
    print(f"  {n:5d} {a:>12s} {b:>18s} {yv[0]:8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.3))
w = 0.35
vals = {"set transformer": [mse(att, te, yte), mse(att, te_sh, yte)],
        "MLP on flattened": [mse(mlp, te, yte), mse(mlp, te_sh, yte)]}
for k, (name, col) in enumerate([("set transformer", GEO_TEAL), ("MLP on flattened", GEO_RUST)]):
    axes[0].bar(np.arange(2) + (k - 0.5) * w, vals[name], w, label=name, color=col)
axes[0].axhline(base, color="0.4", ls="--", lw=1.2, label="predict the mean")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["as generated", "shuffled"])
axes[0].set_yscale("log"); axes[0].set_ylabel("test MSE")
axes[0].set_title("the label is unchanged by shuffling"); axes[0].legend(fontsize=7.5)

pa, pm = predict(att, te), predict(mlp, te_sh)
axes[1].scatter(yte, pa, s=5, alpha=0.4, color=GEO_TEAL, label="transformer (shuffled ok)")
axes[1].scatter(yte, pm, s=5, alpha=0.4, color=GEO_RUST, label="MLP on shuffled")
lims = [yte.min(), yte.max()]
axes[1].plot(lims, lims, "k--", lw=1)
axes[1].set_xlabel("true diameter"); axes[1].set_ylabel("predicted")
axes[1].legend(fontsize=7.5); axes[1].set_title("predictions")
plt.tight_layout(); plt.show()

The transformer wins outright, on both counts that matter. It is an order of
magnitude more accurate than the padded MLP while carrying a third of the parameters,
and it is **exactly** unaffected by shuffling — not approximately but to the last
digit, because permutation invariance is an algebraic property of `softmax(QK^T)V`
followed by a mean, not something it learned. Not one of its predictions moves when
the points are reordered; every one of the MLP's does. It also takes clouds far longer
than anything it trained on, while the MLP simply refuses: its input width was fixed
the moment it was constructed.

The MLP tells the more interesting story about *why*. On sorted input it is
respectable — with points ordered by angle, the diameter pair sits at roughly opposite
indices, and a dense network can exploit that regularity. Shuffle the points and the
regularity evaporates. Nothing about the *geometry* changed — same points, same
diameter — only the labelling, which was never information in the first place.

This is Tutorial 5's lesson with a different group. There the symmetry was
translation and the fix was convolution; here it is $S_n$ and the fix is attention
(or, more cheaply, any pooling over a shared per-point map — the DeepSets
construction). Lecture 10 gives the general statement.

> **Exercise 1 — how much structure do you actually need?**
> (a) Replace attention by **DeepSets**: embed each point with a shared MLP, sum
> over points, apply a head. That is permutation-invariant too, and has no
> $n\times n$ interaction. How close does it get on the diameter, and why might it
> struggle?
>
> (b) Sort the points (by angle, say) before feeding the MLP. Sorting is a cheap way
> to make a set canonical. Does it fix the MLP, and what has it cost you?
>
> (c) Change the label to the **mean pairwise distance** instead of the maximum. One
> of these tasks needs to identify a specific pair and one does not — which model's
> advantage shrinks?

---
## 4. What is the model attending to?

The matrix $A$ is interpretable in a way that most learned objects are not: row $i$
is an honest probability distribution over points, and we can plot it. For a
diameter task there is a clear hypothesis to test — the model should care about
the extreme points and ignore the interior.

We measure this directly. For each cloud, compute how much total attention each
point receives (the column sums of $A$), and compare that with the point's
**eccentricity**: its distance from the cloud's centroid.

In [ ]:
NS = 400
recv, ecc, is_extreme = [], [], []
with torch.no_grad():
    for C in te[:NS]:
        _, A = att(torch.tensor(C)[None], return_A=True)   # one cloud, at its own n
        A = A[0].numpy()
        recv.append(A.sum(axis=0))                         # attention received per point
        ecc.append(np.linalg.norm(C - C.mean(0), axis=-1))  # distance from the centroid
        e = np.zeros(len(C), bool); e[list(diameter_pair(C))] = True
        is_extreme.append(e)

recv = np.concatenate(recv); ecc = np.concatenate(ecc); is_extreme = np.concatenate(is_extreme)

corr = np.corrcoef(ecc, recv)[0, 1]
print(f"correlation(attention received, distance from centroid) = {corr:+.3f}")
print(f"mean attention received by the diameter pair : {recv[is_extreme].mean():.3f}")
print(f"mean attention received by all other points : {recv[~is_extreme].mean():.3f}")
print(f"ratio: {recv[is_extreme].mean() / recv[~is_extreme].mean():.2f}x")

In [ ]:
fig = plt.figure(figsize=(11.2, 3.2))

ax = fig.add_subplot(131)
ax.scatter(ecc, recv, s=3, alpha=0.15, color=GEO_DARK)
ax.set_xlabel("distance from centroid"); ax.set_ylabel("attention received")
ax.set_title(f"correlation {corr:+.2f}")

C0 = te[0]
with torch.no_grad():
    _, A0 = att(torch.tensor(C0)[None], return_A=True)
A0 = A0[0].numpy()

ax = fig.add_subplot(132)
im = ax.imshow(A0, cmap="magma")
ax.set_xlabel("attends to"); ax.set_ylabel("point")
ax.set_title(f"one attention matrix $A$  ($n = {len(C0)}$)"); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.85)

ax = fig.add_subplot(133)
i, j = diameter_pair(C0)
sc = ax.scatter(*C0.T, c=A0.sum(axis=0), s=90, cmap="magma", zorder=3)
ax.plot(C0[[i, j], 0], C0[[i, j], 1], color=GEO_TEAL, lw=2, zorder=2)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("attention received (colour)\nteal: the diameter pair")
fig.colorbar(sc, ax=ax, shrink=0.85)
plt.tight_layout(); plt.show()

Read the numbers before the pictures — and here they support the hypothesis. The
diameter pair receives several times the attention of an average point, and
attention received correlates positively with distance from the centroid. The model
has, without being told, learned to look at the extremes.

That is a genuinely satisfying result, so it is the right moment to say why it is
not proof. Attention maps are the most over-interpreted objects in modern machine
learning: they are inspectable, which tempts one to read them as explanations, and
there is a substantial literature showing that attention weights and causal
importance can come apart. What we have measured is a *correlation* between
attention and a quantity we believe matters. What would settle it is an
intervention — suppress the attention paid to those points and see whether the
prediction actually degrades. That is Exercise 2(a), and you should do it before
believing this figure.

The discipline is the same as in Tutorial 5 §4: state a hypothesis about what the
model should attend to, compute a number that could refute it, and then check that
the number is measuring what you think.

> **Exercise 2 — attention, tested rather than admired.**
> (a) **Ablation.** Zero out the attention a cloud pays to its two extreme points and
> measure how much the prediction changes; compare with zeroing two random points. If
> the extremes matter causally, the two should differ.
>
> (b) Train on the **mean** pairwise distance instead of the diameter and recompute
> the ratio above. A task with no special pair should show even less concentration —
> does it?
>
> (c) Add a second attention layer and re-measure. Does depth sharpen the attention
> onto the relevant pair, and does test error improve alongside it?

> **Exercise 3 — when order does matter.**
> Attention alone cannot see order, which is why transformers add positional
> encodings. Build a task where order carries the answer: treat each cloud as the
> vertex sequence of a closed polygon and predict its **signed area**
> $\frac12\sum_i (x_i y_{i+1} - x_{i+1} y_i)$, which changes sign if you reverse the
> traversal. Show that the set model cannot do better than chance on the sign, then
> switch on `positional=True` — the sinusoidal encoding of slide 5 is already built —
> and watch it become solvable. (On the diameter it buys nothing, as it should not.)

---
## 5. What to take away

- Attention is a **data-dependent averaging operator**: the same shape as a
  convolution, but with the neighbourhood computed from the input rather than fixed
  by a grid. That is what lets it act on sets, graphs, and simplicial data — of
  whatever size each input happens to be, since $n$ enters only as a dimension and
  the pooling contracts it away. **Attention never needs padding** — not for the model,
  and not for batching once equal lengths are grouped. Only fixed-width architectures
  do, which is one more way of saying they have the wrong symmetry.
- Its native symmetry is **permutation**, and the invariance is algebraic — a model
  built from attention plus a mean is exactly invariant, not approximately.
- **The symmetry mismatch is the whole story.** An MLP on a flattened cloud extracts
  information from an ordering that carries none, and pays for it the moment the
  ordering changes.
- Attention needs **positional encoding** precisely because it cannot see order. When
  the geometry does have an order — a filtration, a boundary traversal — you must
  supply it as a feature.
- **Attention maps are evidence, not explanations.** Test them with ablations before
  believing the picture.

### Next

**Lecture 7** leaves supervised learning behind: with no labels at all, what
structure can be recovered from a point cloud? **Tutorial 7** uses clustering to
detect connected components and geometric phases.

### Further reading

- Vaswani et al., "Attention is all you need", *NeurIPS* 2017 — the original.
- Zaheer et al., "Deep Sets", *NeurIPS* 2017 — the minimal permutation-invariant architecture of Exercise 1(a).
- Lee et al., "Set Transformer", *ICML* 2019 — attention designed for sets, and the model §3 is a stripped-down version of.
- Jain & Wallace, "Attention is not explanation", *NAACL* 2019 — the caution in §4.